In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial
import gc

# local imports
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.CutMasks.CutMasks import *

from makedf.mcstat import get_MCstat_unc

from analysis_village.cc1pi.var_configs import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result
save_fig_dir = "/exp/sbnd/data/users/lpelegri/Graphs/syst/detector"

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_fixdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

# BNB data
# data_tot_pot = data_hdr_df['TOR875'].sum()
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
pot_str = f"{data_tot_pot} $\\times 10^{18}$"
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))


In [ ]:

#Load systematics df
#syst_names = ["2xSCE", "PMTGainFluct","PMTHighNoise", "PMTLowEff", "CCalVar", "$C_{cal}$ Variation"]

syst_keys = ["SystVarsCV","wiremod_YZ","wiremod_XZ_thetaXW", "0xSCE","2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14","C15","C16"]
labels    = ["CV", r"Wiremod Y-Z",r"Wiremod X$\theta_{xw}$", "0xSCE", "2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff", "ccalm", "ccalp", "alpham", "alphap", "rm", "rp", "betam", "betap"]

'''
syst_keys = ["SystVarsCV","wiremod_YZ", "ccalm", "ccalp"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6","C7","C8","C9","C10","C11","C12","C13","C14","C15","C16"]
labels    = ["CV", r"Wiremod Y-Z", "ccalm", "ccalp"]
'''

detvar_plotter = partial(
    variation_hists,
    var_colors=colors,
    var_labels=labels,
    approval="internal"
)

detvar_plotter_final_vars = partial(
    variation_hists_final_vars,
    var_colors=colors,
    var_labels=labels,
    approval="internal"
)

keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
syst_dfs = {}
syst_evt_dfs = {}
for key in syst_keys:
    print(f"--- Processing {key} ---")
    
    # 1. Load (Temporary)
    temp_df = load_df(f"/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_{key}.df", keys2load, 100)
    
    # 2. Scale POT
    syst_tot_pot = temp_df['hdr']['pot'].sum()
    syst_pot_scale = data_tot_pot / syst_tot_pot
    temp_df['cc1pi'][pot_weight_col] = syst_pot_scale
    
    # 3. Process & Match
    # This overwrites the slice-level DF with the matched version
    matched_evt = perform_truth_matching(temp_df['cc1pi'], temp_df['nudf'])

    # 5. Store only the reduced result
    syst_evt_dfs[key] = matched_evt
    
    # 6. Aggressive Cleanup
    del temp_df
    gc.collect()

# Prepare maks

In [ ]:
# --- CV ---
#proton sideband
'''
syst_evt_sideband_dfs = {}
for key, df in syst_evt_dfs.items():
    print(key)
    syst_masks = build_event_cumulative_masks(df, plot_sideband = False)["energy"]
    syst_sideband_masks = build_event_cumulative_masks(df, plot_sideband = True)["energy"]
    
    syst_evt_sideband_dfs[key] = syst_evt_dfs[key][syst_sideband_masks]
    syst_evt_dfs[key] = syst_evt_dfs[key][syst_masks]
'''

#Big sideband
syst_evt_sideband_dfs = {}
for key, df in syst_evt_dfs.items():
    print(key)
    normal_masks = build_event_cumulative_masks(df, sideband = "")["energy"]
    shower_mask = build_event_cumulative_masks(df, sideband = "shower")["energy"]
    proton_mask = build_event_cumulative_masks(df, sideband = "proton")["energy"]
    sideband_mask = shower_mask | proton_mask
    
    syst_evt_sideband_dfs[key] = syst_evt_dfs[key][sideband_mask]    
    syst_evt_dfs[key] = syst_evt_dfs[key][normal_masks]


# Get all the syst uncert

In [ ]:
from analysis_village.cc1pi.HelperFunctions import HelperFunctions
for key in syst_evt_dfs:
    HelperFunctions.print_purity(syst_evt_dfs[key], ('truth','nu_categ','','','',''))
    HelperFunctions.print_purity(syst_evt_sideband_dfs[key], ('truth','nu_categ','','','',''))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

unisim_keys = []
paired_syst = {} 
file_dir = "/exp/sbnd/data/users/lpelegri/syst/CCBC_rates_proton_shower_sideband"
os.makedirs(file_dir, exist_ok=True)

if 'syst_dict' not in locals():
    syst_dict = {}

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]

# Identify Unisim (1 univ) vs Paired/Multisim (2 univ)
for key in syst_keys:
    if key.endswith('m') or key.endswith('p'):
        base = key[:-1]
        if base not in paired_syst:
            paired_syst[base] = [None, None]
        if key.endswith('m'): paired_syst[base][0] = key
        else: paired_syst[base][1] = key
    else:
        unisim_keys.append(key)

print(unisim_keys)
univ_hist_detector = {}

# Initialize structure for all systematics
for syst_key in unisim_keys:
    univ_hist_detector[syst_key] = {"Ps": {}, "Bs": {}, "nc": {}}

for base_name in paired_syst:
    univ_hist_detector[base_name] = {"Ps": {}, "Bs": {}, "nc": {}}

# --- 3. Main Variable Loop ---
for var_config in var_configs:
    var_name = var_config.var_save_name
    print(var_name)
    evtdfs_signal = [syst_evt_dfs[syst_key][syst_evt_dfs[syst_key].truth.nu_categ == "CC1pi"].groupby(level=['__ntuple', 'entry', 'rec.slc..index'], sort=False).first() for syst_key in syst_evt_dfs.keys()]
    evtdfs_bkg = [syst_evt_dfs[syst_key][syst_evt_dfs[syst_key].truth.nu_categ != "CC1pi"].groupby(level=['__ntuple', 'entry', 'rec.slc..index'], sort=False).first() for syst_key in syst_evt_dfs.keys()]
    sideband_evtdfs = [syst_evt_sideband_dfs[syst_key].groupby(level=['__ntuple', 'entry', 'rec.slc..index'], sort=False).first() for syst_key in syst_evt_dfs.keys()]

    n_phis = detvar_plotter_final_vars(evtdfs_signal, var_name=var_config.var_evt_reco_col, bins=var_config.bins, plot=False)
    n_Bs = detvar_plotter_final_vars(evtdfs_bkg, var_name=var_config.var_evt_reco_col, bins=var_config.bins, plot=False)
    n_sideband = detvar_plotter_final_vars(sideband_evtdfs, var_name=var_config.var_evt_reco_col, bins=var_config.bins, plot=False)
    
    key_to_idx = {key: i for i, key in enumerate(syst_evt_dfs.keys())}
    for syst_key in unisim_keys:
        kidx = key_to_idx[syst_key]
        # This creates (1, nbins)
        univ_hist_detector[syst_key]["Ps"][var_name] = np.array(n_phis[kidx], ndmin=2)
        univ_hist_detector[syst_key]["Bs"][var_name] = np.array(n_Bs[kidx], ndmin=2)
        univ_hist_detector[syst_key]["nc"][var_name] = np.array(n_sideband[kidx], ndmin=2)
     
    for base_name, keys in paired_syst.items():
        m_idx, p_idx = key_to_idx.get(keys[0]), key_to_idx.get(keys[1])
        if m_idx is not None and p_idx is not None:
            # This creates (2, nbins) -> Exactly like univ_events
            univ_hist_detector[base_name]["Ps"][var_name] = np.stack([n_phis[m_idx], n_phis[p_idx]])
            univ_hist_detector[base_name]["Bs"][var_name] = np.stack([n_Bs[m_idx], n_Bs[p_idx]])
            univ_hist_detector[base_name]["nc"][var_name] = np.stack([n_sideband[m_idx], n_sideband[p_idx]])

# --- CRITICAL PART FOR SAVING ---
# np.savez does not like deep nesting. To preserve your dict structure:
save_path = os.path.join(file_dir, "detector_univ_hists.npz")
np.savez(save_path, **{k: np.array(v, dtype=object) for k, v in univ_hist_detector.items()})